# Step 5.2: strategy versions

This notebook is the check of step 5.2. A strategy you keep is now saved as a version: its code, a module whose function `strategy` gives the `SignalStrategy`, with a hypothesis, one sentence stating the edge, and a changelog, in a file written once and never changed, which the app lists too. Here you save the breakout template as the strategy `check_breakout`, save a second version with a wider stop, read the diff of their code, and see a version with a syntax error refused at its line.

Run All from the top. The notebook reads none of your market data. It writes two small files, `strategies/check_breakout/v1.json` and `v2.json`, in your workspace under `FEFAT_HOME`, which stay; running it again reads them back and writes nothing.

## Two versions

The breakout template saved as version 1, with the hypothesis its docstring states, then with a stop of 2 ATRs by default as version 2.

Expected: `check_breakout v1 and v2 on MES 5m`.

In [ ]:
import fefat as ff

breakout = next(each for each in ff.strategy.templates() if each.name == "breakout")
first = ff.strategy.save(
    "check_breakout", breakout.code, hypothesis=breakout.hypothesis, changelog="The template."
)
wider = breakout.code.replace("stop_atr: float = 1.5", "stop_atr: float = 2.0")
second = ff.strategy.save(
    "check_breakout", wider, hypothesis=breakout.hypothesis, changelog="A wider stop."
)
print(
    f"{first.name} v{first.version} and v{second.version} on {second.instrument} {second.timeframe}"
)

## Their diff

The unified diff of the two versions' code, which the app's diff gives too: the line of the stop's default, with `-` for version 1 and `+` for version 2.

Expected: the diff's header, then the `def strategy` line twice, with `1.5` and `2.0`.

In [ ]:
print(ff.strategy.diff("check_breakout", 1, 2))

## A syntax error

A version whose code does not parse is refused with the line it is at, which the app's editor underlines, before anything is written.

Expected: `Refused at line 10: SyntaxError: invalid syntax`.

In [ ]:
broken = breakout.code.replace('close = pl.col("close")', 'close = pl.col("close") +')
try:
    ff.strategy.save("check_breakout", broken, hypothesis=breakout.hypothesis)
    refused_at = None
except ff.strategy.StrategyCodeError as refused:
    refused_at = refused.line
    print(f"Refused at line {refused.line}: {refused}")

## The verdict

Expected: `PASSED` three times.

In [ ]:
kept = [ff.strategy.load("check_breakout", number) for number in (1, 2)]
changelogs = [version.changelog for version in kept]
changed = "+def strategy(lookback: int = 12, stop_atr: float = 2.0)"
checks = {
    "Two versions saved": changelogs == ["The template.", "A wider stop."],
    "The diff shows the wider stop": changed in ff.strategy.diff("check_breakout", 1, 2),
    "The syntax error is refused at its line": refused_at == 10,
}
for check, passed in checks.items():
    print(f"{check}: {'PASSED' if passed else 'FAILED'}")